# NYXARA on Kaggle — full run

Run NYXARA's full stack on Kaggle: her PRIMARY brain AiCredits (a cloud tool she
controls) with her own local `self`/`native` brains as the sovereign floor,
persistent memory saved to the notebook output, an optional phone-browser chat via a Cloudflare
tunnel, and (optional) the real LoRA forge of her own brain.

## Before running — notebook settings (right sidebar → ⚙ Settings)
1. **Accelerator:** `None (CPU)` is fine — her primary model runs in the cloud; a GPU only speeds the optional LoRA forge of her OWN brain.
2. **Internet:** `On` (needed to reach the cloud tools)
3. Phone verification must be done once on your Kaggle account to unlock GPU + internet.

Then just **Run All**. Cells are ordered: setup → chat → save → (optional) phone server → (optional) training.

In [ ]:
# 1) Get the code -------------------------------------------------------------
import os

REPO_DIR = "/kaggle/working/NYXARAv01"
if not os.path.exists(REPO_DIR):
    # Public repo:
    !git clone --depth 1 https://github.com/nyxarajp-del/NYXARAv01 {REPO_DIR}
    # Private repo? Save a GitHub token in Add-ons → Secrets as GITHUB_TOKEN, then use:
    #   from kaggle_secrets import UserSecretsClient
    #   tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    #   os.system(f"git clone --depth 1 https://{tok}@github.com/nyxarajp-del/NYXARAv01 {REPO_DIR}")
%cd {REPO_DIR}

In [ ]:
# 2) Install — her on-device primary + the cloud tools + the LoRA foundry ------
# .[litertlm] installs the LiteRT-LM runtime that serves her PRIMARY brain in-process (the
# ~2.4 GB weights come in the next cell); the `openai` SDK (via .[llm]) reaches her cloud rungs
# (aicredits/groq/airouter); peft (via .[foundry]) lets the foundry LoRA-fine-tune its own base
# into her OWN forged brain — served through the `self` provider.
%pip install -q -e ".[litertlm,llm,foundry,reasoning,security,observe]"

import transformers, torch
print("transformers:", transformers.__version__, "| torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available())

In [ ]:
# 3) Persistence — NYXARA's memory survives the session -----------------------
# Everything under /kaggle/working is downloadable / saved with the notebook output,
# so both her config home (NYXARA_HOME) and the console snapshot dir (~/.nyxara)
# point there. Re-running the notebook later restores her memory.
import os, shutil

NYX_HOME = "/kaggle/working/nyxara-home"
os.makedirs(NYX_HOME, exist_ok=True)
os.environ["NYXARA_HOME"] = NYX_HOME

link = os.path.expanduser("~/.nyxara")
if os.path.isdir(link) and not os.path.islink(link):
    shutil.copytree(link, NYX_HOME, dirs_exist_ok=True)
    shutil.rmtree(link)
if not os.path.islink(link):
    os.symlink(NYX_HOME, link)
print("NYXARA_HOME →", NYX_HOME)

In [ ]:
# 4) Configure the brain — her ON-DEVICE primary, the cloud beneath it ---------------------
# Her PRIMARY model is `litertlm`: Gemma-4-E2B-it served in-process, no key and no network.
# The `auto` ladder is litertlm → aicredits → groq → airouter → self → native. When a rung is
# unreachable the next one answers, and when nothing above is reachable her own always-on native
# own-brain does — she is never dependent on the cloud, and no raw third-party model ever speaks
# as her.
import os

os.environ.update({
    "NYXARA_LLM__PROVIDER": "auto",
    "NYXARA_LLM__AICREDITS_MODEL": "moonshotai/kimi-k2-thinking",
    # her on-device primary — the next cell fetches its weights
    "NYXARA_LLM__LITERTLM_ENABLED": "true",
    "NYXARA_LLM__LITERTLM_AUTO_DOWNLOAD": "true",
    # MAXIMUM POWER — crank every capability/depth/cadence knob to its ceiling. Safety
    # boundaries (invariants/audit/corrigibility/soul-binding, simulation + sandbox) stay
    # forced on; this only turns every capability on. Set to "false" for a lean posture.
    "NYXARA_MAX_POWER": "true",
})
print("provider:", os.environ["NYXARA_LLM__PROVIDER"],
      "| model:", os.environ["NYXARA_LLM__AICREDITS_MODEL"],
      "| max_power:", os.environ["NYXARA_MAX_POWER"])

In [ ]:
# 5) Fetch her primary brain — ~2.4 GB, once per session ----------------------
# Without it the rung is honestly unavailable and the ladder starts at aicredits;
# with it she answers on her own hardware, key-free and network-free.
!python scripts/fetch_litertlm_model.py

from nyxara.mind.llm import LLM
from nyxara.kernel.config import get_settings

_llm = LLM(settings=get_settings())
print("provider status:", _llm.provider_status())
print("drafting on    :", _llm.chosen_provider().name)


In [ ]:
# 5) Boot NYXARA + a chat helper ----------------------------------------------
# Her primary model answers from the cloud — no model download on boot.
from nyxara.agency.permissions import Authority
from nyxara.kernel.orchestrator import NyxaraCore

core = NyxaraCore()
restored = core.load_state()
if restored:
    print(f"continuity: restored {restored} memories from a prior session")

def ask(text: str) -> None:
    """One full turn of the sovereign cognitive cycle, as the Master."""
    result = core.process(text, authority=Authority.OWNER)
    print("NYXARA:", result.response or result.reason or "(no response)")

ask("Hello NYXARA — report your status.")

In [ ]:
# Talk to her — edit the text and re-run this cell as many times as you like.
ask("What can you do for me?")

In [ ]:
# 6) Save memory before the session ends --------------------------------------
# (Run this any time; also run it last. The snapshot lands in /kaggle/working/nyxara-home,
# which Kaggle keeps as notebook output — download it or let the next run restore it.)
path = core.save_state()
print("memory persisted →", path)

## Optional A — chat from your phone's browser 📱

Starts NYXARA's FastAPI server (`nyxara-serve`) inside Kaggle and exposes it through a free
Cloudflare quick-tunnel. The cell prints a `https://….trycloudflare.com` URL — open it on your
phone. Keep the cell running; every request needs the `Authorization: Bearer <token>` header.
**Change the token below before running.**

In [ ]:
%pip install -q -e ".[server]"
import os, subprocess, time

os.environ["NYXARA_SERVER__API_TOKEN"] = "CHANGE-ME-strong-token"
server = subprocess.Popen(["nyxara-serve"], env=os.environ.copy())
time.sleep(10)  # let the server boot

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
# Blocks while the tunnel is up — the public URL is printed a few lines below.
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate

## Optional B — forge her OWN brain (DistilGPT-2 LoRA) 🔥

The real forge: LoRA-tunes the DistilGPT-2 base on her identity corpus and promotes the
adapter so `NYXARA_LLM__PROVIDER=self` serves it. Notes for Kaggle:

- Downloads the ~350 MB base into the HF cache (session disk, not /kaggle/working).
- At ~82M params it trains full-precision LoRA on CPU or GPU — no quantization needed.
- The promoted adapter lands under `nyxara-home/` (small, MBs) → persisted with the output, so
  you can download it and serve it anywhere later.
- With `.[foundry]` present this is the REAL LoRA; without it the forge degrades to her
  always-on n-gram brain rather than crashing.

In [ ]:
!GEN=1 bash scripts/lora_tune_distilgpt2.sh

# After the forge promotes her brain, serve it as primary:
#   os.environ["NYXARA_LLM__PROVIDER"] = "self"
#   then re-run the boot cell (5) above.